# 07_03C — LightGBM + XGBoost + Category Encoders — Early-Stage

Este notebook substitui a versão com CatBoost nativo por alternativas mais compatíveis com ambiente corporativo:

- LightGBM, se disponível;
- XGBoost, se disponível;
- `category_encoders.CatBoostEncoder`, que é apenas um encoder supervisionado e **não exige instalar CatBoost**;
- fallback com `HistGradientBoostingClassifier` do scikit-learn.

## Target correto

```text
0 = banco ganhou
1 = banco perdeu
```

Logo, a classe positiva é a perda:

```text
proba_perda = predict_proba[:, 1]
```

## Entradas esperadas

```text
data/processed/abt_early_stage_v1_dev_hist.parquet
data/processed/abt_early_stage_v1_holdout_hist.parquet
data/processed/feature_list_early_stage_v1_hist.txt
```

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import importlib.util

warnings.filterwarnings("ignore")

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

pd.set_option("display.max_columns", 400)
pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 260)

BASE_DIR = Path("..")
PROCESSED_DIR = BASE_DIR / "data" / "processed"
REPORTS_DIR = BASE_DIR / "outputs" / "reports" / "modelagem_early_stage"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

DEV_HIST_FILE = PROCESSED_DIR / "abt_early_stage_v1_dev_hist.parquet"
HOLDOUT_HIST_FILE = PROCESSED_DIR / "abt_early_stage_v1_holdout_hist.parquet"
FEATURE_LIST_HIST_FILE = PROCESSED_DIR / "feature_list_early_stage_v1_hist.txt"

TARGET_COL_LEGACY = "target_banco_ganhou"
TARGET_COL = "target_banco_perdeu"
DATE_COL = "Datainicial"
VALUE_COL = "fe_valor_ajuizado"
RANDOM_STATE = 42

print("Setup concluído.")
print("Semântica: 0=banco ganhou | 1=banco perdeu")

## 2. Verificar bibliotecas disponíveis

In [ ]:
HAS_LIGHTGBM = importlib.util.find_spec("lightgbm") is not None
HAS_XGBOOST = importlib.util.find_spec("xgboost") is not None
HAS_CATEGORY_ENCODERS = importlib.util.find_spec("category_encoders") is not None
HAS_FEATURE_ENGINE = importlib.util.find_spec("feature_engine") is not None

print("lightgbm disponível:", HAS_LIGHTGBM)
print("xgboost disponível:", HAS_XGBOOST)
print("category_encoders disponível:", HAS_CATEGORY_ENCODERS)
print("feature_engine disponível:", HAS_FEATURE_ENGINE)

if HAS_LIGHTGBM:
    from lightgbm import LGBMClassifier

if HAS_XGBOOST:
    from xgboost import XGBClassifier

if HAS_CATEGORY_ENCODERS:
    import category_encoders as ce

if HAS_FEATURE_ENGINE:
    from feature_engine.encoding import RareLabelEncoder

## 3. Carregar bases

In [ ]:
df_dev = pd.read_parquet(DEV_HIST_FILE)
df_holdout = pd.read_parquet(HOLDOUT_HIST_FILE)

with open(FEATURE_LIST_HIST_FILE, "r", encoding="utf-8") as f:
    feature_list = [line.strip() for line in f if line.strip()]

df_dev[DATE_COL] = pd.to_datetime(df_dev[DATE_COL], errors="coerce")
df_holdout[DATE_COL] = pd.to_datetime(df_holdout[DATE_COL], errors="coerce")

df_dev[TARGET_COL] = df_dev[TARGET_COL_LEGACY].astype(int)
df_holdout[TARGET_COL] = df_holdout[TARGET_COL_LEGACY].astype(int)

feature_list = [c for c in feature_list if c in df_dev.columns and c in df_holdout.columns]

print("Dev:", df_dev.shape)
print("Holdout:", df_holdout.shape)
print("Features disponíveis:", len(feature_list))
display(df_dev.head(3))

## 4. Funções auxiliares

In [ ]:
def save_report(df_report, filename):
    path = REPORTS_DIR / filename
    df_report.to_csv(path, index=False)
    print(f"Relatório salvo em: {path}")


def target_distribution(df, name):
    out = df[TARGET_COL].value_counts(dropna=False).rename_axis("target").reset_index(name="qtd")
    out["classe"] = out["target"].map({0: "banco_ganhou", 1: "banco_perdeu"})
    out["perc"] = out["qtd"] / out["qtd"].sum()
    out["dataset"] = name
    return out


def infer_feature_types(df, features):
    numeric_features, categorical_features, text_features = [], [], []
    for col in features:
        if col == "fe_texto_inicial_curto":
            text_features.append(col)
        elif pd.api.types.is_numeric_dtype(df[col]):
            numeric_features.append(col)
        else:
            categorical_features.append(col)
    return numeric_features, categorical_features, text_features


def make_temporal_folds(df, date_col=DATE_COL):
    folds = [(0.55, 0.70), (0.70, 0.85), (0.85, 1.00)]
    df_sorted = df.sort_values(date_col).copy()
    dates = df_sorted[date_col]
    out = []
    for i, (train_end_q, val_end_q) in enumerate(folds, start=1):
        train_end_date = dates.quantile(train_end_q)
        val_end_date = dates.quantile(val_end_q)
        train_idx = df_sorted.index[df_sorted[date_col] <= train_end_date]
        val_idx = df_sorted.index[(df_sorted[date_col] > train_end_date) & (df_sorted[date_col] <= val_end_date)]
        out.append({
            "fold": i,
            "train_start_date": df_sorted.loc[train_idx, date_col].min(),
            "train_end_date": train_end_date,
            "val_start_date": df_sorted.loc[val_idx, date_col].min(),
            "val_end_date": val_end_date,
            "train_idx": train_idx,
            "val_idx": val_idx,
            "qtd_train": len(train_idx),
            "qtd_val": len(val_idx),
        })
    return out


def threshold_metrics_perda(y_true, proba_perda, threshold=0.5):
    pred = (proba_perda >= threshold).astype(int)
    return {
        "threshold": threshold,
        "precision_perda": precision_score(y_true, pred, zero_division=0),
        "recall_perda": recall_score(y_true, pred, zero_division=0),
        "f1_perda": f1_score(y_true, pred, zero_division=0),
        "pred_perda_rate": pred.mean(),
    }


def find_best_f1_threshold_perda(y_true, proba_perda):
    precision, recall, thresholds = precision_recall_curve(y_true, proba_perda)
    if len(thresholds) == 0:
        return 0.5, 0, 0, 0
    precision_t = precision[:-1]
    recall_t = recall[:-1]
    f1_t = 2 * precision_t * recall_t / np.maximum(precision_t + recall_t, 1e-12)
    best_idx = np.nanargmax(f1_t)
    return thresholds[best_idx], precision_t[best_idx], recall_t[best_idx], f1_t[best_idx]


def topk_risco_perda_metrics(y_true, proba_perda, ks=(0.01, 0.05, 0.10, 0.20)):
    df_tmp = pd.DataFrame({"y_true": y_true, "score_risco_perda": proba_perda})
    df_tmp = df_tmp.sort_values("score_risco_perda", ascending=False).reset_index(drop=True)
    total_perdas = (df_tmp["y_true"] == 1).sum()
    taxa_perda_base = (df_tmp["y_true"] == 1).mean()
    out = []
    for k in ks:
        n = max(1, int(np.ceil(len(df_tmp) * k)))
        top = df_tmp.head(n)
        precision_k = (top["y_true"] == 1).mean()
        recall_k = (top["y_true"] == 1).sum() / total_perdas if total_perdas else np.nan
        lift_k = precision_k / taxa_perda_base if taxa_perda_base else np.nan
        out.append({
            "top_k": k,
            "n_top": n,
            "precision_perda_at_k": precision_k,
            "recall_perda_at_k": recall_k,
            "lift_perda_at_k": lift_k,
            "taxa_perda_base": taxa_perda_base,
        })
    return pd.DataFrame(out)


def topk_prioridade_financeira_metrics(y_true, proba_perda, valor_ajuizado, ks=(0.01, 0.05, 0.10, 0.20)):
    df_tmp = pd.DataFrame({
        "y_true": y_true,
        "proba_perda": proba_perda,
        "valor_ajuizado": pd.to_numeric(valor_ajuizado, errors="coerce").fillna(0),
    })
    df_tmp["score_prioridade_financeira"] = df_tmp["proba_perda"] * df_tmp["valor_ajuizado"]
    df_tmp = df_tmp.sort_values("score_prioridade_financeira", ascending=False).reset_index(drop=True)
    total_perdas = (df_tmp["y_true"] == 1).sum()
    taxa_perda_base = (df_tmp["y_true"] == 1).mean()
    total_valor = df_tmp["valor_ajuizado"].sum()
    total_valor_perdas = df_tmp.loc[df_tmp["y_true"] == 1, "valor_ajuizado"].sum()
    out = []
    for k in ks:
        n = max(1, int(np.ceil(len(df_tmp) * k)))
        top = df_tmp.head(n)
        valor_top = top["valor_ajuizado"].sum()
        valor_perdas_top = top.loc[top["y_true"] == 1, "valor_ajuizado"].sum()
        precision_k = (top["y_true"] == 1).mean()
        recall_k = (top["y_true"] == 1).sum() / total_perdas if total_perdas else np.nan
        lift_k = precision_k / taxa_perda_base if taxa_perda_base else np.nan
        out.append({
            "top_k": k,
            "n_top": n,
            "precision_perda_at_k": precision_k,
            "recall_perda_at_k": recall_k,
            "lift_perda_at_k": lift_k,
            "taxa_perda_base": taxa_perda_base,
            "valor_ajuizado_top": valor_top,
            "share_valor_ajuizado_total": valor_top / total_valor if total_valor else np.nan,
            "valor_ajuizado_perdas_top": valor_perdas_top,
            "share_valor_perdas_total": valor_perdas_top / total_valor_perdas if total_valor_perdas else np.nan,
        })
    return pd.DataFrame(out)


def get_positive_class_proba(model, X):
    proba = model.predict_proba(X)
    return proba[:, 1] if getattr(proba, "ndim", 1) > 1 else proba


def to_1d_text(x):
    # ColumnTransformer envia texto como DataFrame/array 2D; TfidfVectorizer espera 1D.
    if isinstance(x, pd.DataFrame):
        return x.iloc[:, 0].fillna("").astype(str).values
    x = np.asarray(x)
    if x.ndim == 2:
        return pd.Series(x[:, 0]).fillna("").astype(str).values
    return pd.Series(x).fillna("").astype(str).values

## 5. Distribuição do target e tipos de features

In [ ]:
target_dist = pd.concat([
    target_distribution(df_dev, "dev"),
    target_distribution(df_holdout, "holdout"),
], ignore_index=True)

save_report(target_dist, "66_3c_lgbm_xgb_target_distribution.csv")
display(target_dist)

numeric_features, categorical_features, text_features = infer_feature_types(df_dev, feature_list)

feature_type_summary = pd.DataFrame([
    {"tipo": "numeric", "qtd": len(numeric_features)},
    {"tipo": "categorical", "qtd": len(categorical_features)},
    {"tipo": "text", "qtd": len(text_features)},
])
save_report(feature_type_summary, "67_3c_lgbm_xgb_feature_type_summary.csv")
feature_type_summary

## 6. Selecionar features

In [ ]:
USE_TEXT_FEATURES = False

if not USE_TEXT_FEATURES:
    selected_text_features = []
    selected_features = [c for c in feature_list if c not in text_features]
else:
    selected_text_features = text_features
    selected_features = feature_list.copy()

selected_numeric_features = [c for c in numeric_features if c in selected_features]
selected_categorical_features = [c for c in categorical_features if c in selected_features]

print("USE_TEXT_FEATURES:", USE_TEXT_FEATURES)
print("Features selecionadas:", len(selected_features))
print("Numéricas:", len(selected_numeric_features))
print("Categóricas:", len(selected_categorical_features))
print("Texto:", len(selected_text_features))

## 7. Criar folds temporais

In [ ]:
folds = make_temporal_folds(df_dev, DATE_COL)

fold_summary_rows = []
for fold in folds:
    train_idx = fold["train_idx"]
    val_idx = fold["val_idx"]
    taxa_perda_train = df_dev.loc[train_idx, TARGET_COL].mean()
    taxa_perda_val = df_dev.loc[val_idx, TARGET_COL].mean()
    fold_summary_rows.append({
        "fold": fold["fold"],
        "train_start_date": fold["train_start_date"],
        "train_end_date": fold["train_end_date"],
        "val_start_date": fold["val_start_date"],
        "val_end_date": fold["val_end_date"],
        "qtd_train": fold["qtd_train"],
        "qtd_val": fold["qtd_val"],
        "taxa_perda_train": taxa_perda_train,
        "taxa_perda_val": taxa_perda_val,
        "taxa_ganho_train": 1 - taxa_perda_train,
        "taxa_ganho_val": 1 - taxa_perda_val,
    })

fold_summary = pd.DataFrame(fold_summary_rows)
save_report(fold_summary, "68_3c_lgbm_xgb_walk_forward_folds_summary.csv")
fold_summary

## 8. Pré-processadores

In [ ]:
def make_preprocessor(mode, numeric_features, categorical_features, text_features):
    transformers = []

    if numeric_features:
        transformers.append((
            "num",
            Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]),
            numeric_features,
        ))

    if categorical_features:
        if mode == "ordinal":
            cat_pipe = Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="constant", fill_value="nao_informado")),
                ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1)),
            ])

        elif mode == "onehot":
            cat_pipe = Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="constant", fill_value="nao_informado")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse_output=True)),
            ])

        elif mode.startswith("ce_"):
            if not HAS_CATEGORY_ENCODERS:
                raise ImportError("category_encoders não disponível.")

            encoder_name = mode.replace("ce_", "")

            if encoder_name == "catboost":
                encoder = ce.CatBoostEncoder(
                    cols=categorical_features,
                    random_state=RANDOM_STATE,
                    sigma=0.05,
                    a=1,
                    handle_missing="value",
                    handle_unknown="value",
                )
            elif encoder_name == "james_stein":
                encoder = ce.JamesSteinEncoder(
                    cols=categorical_features,
                    handle_missing="value",
                    handle_unknown="value",
                )
            elif encoder_name == "m_estimate":
                encoder = ce.MEstimateEncoder(
                    cols=categorical_features,
                    m=20,
                    handle_missing="value",
                    handle_unknown="value",
                )
            elif encoder_name == "target":
                encoder = ce.TargetEncoder(
                    cols=categorical_features,
                    smoothing=20,
                    min_samples_leaf=30,
                    handle_missing="value",
                    handle_unknown="value",
                )
            elif encoder_name == "count":
                encoder = ce.CountEncoder(
                    cols=categorical_features,
                    handle_missing="value",
                    handle_unknown=0,
                    normalize=True,
                )
            else:
                raise ValueError(f"Encoder não suportado: {encoder_name}")

            cat_pipe = Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="constant", fill_value="nao_informado")),
                ("to_df", FunctionTransformer(lambda x: pd.DataFrame(x, columns=categorical_features), feature_names_out="one-to-one")),
                ("encoder", encoder),
            ])
        else:
            raise ValueError(f"Modo não suportado: {mode}")

        transformers.append(("cat", cat_pipe, categorical_features))

    if text_features:
        text_col = text_features[0]
        text_pipe = Pipeline(steps=[
            ("to_1d", FunctionTransformer(to_1d_text, validate=False)),
            ("tfidf", TfidfVectorizer(max_features=1500, min_df=20, ngram_range=(1, 2), strip_accents="unicode", lowercase=True)),
        ])
        transformers.append(("txt", text_pipe, [text_col]))

    return ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0.3, verbose_feature_names_out=False)

## 9. Modelos candidatos

In [ ]:
def make_model_candidates():
    candidates = {}

    candidates["hgb_ordinal"] = {
        "preprocess_mode": "ordinal",
        "model": HistGradientBoostingClassifier(
            learning_rate=0.04,
            max_iter=500,
            max_leaf_nodes=31,
            min_samples_leaf=40,
            l2_regularization=1.0,
            early_stopping=True,
            validation_fraction=0.15,
            random_state=RANDOM_STATE,
        )
    }

    if HAS_LIGHTGBM:
        base_lgbm = dict(
            objective="binary",
            boosting_type="gbdt",
            n_estimators=700,
            learning_rate=0.035,
            num_leaves=31,
            max_depth=-1,
            min_child_samples=50,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.2,
            reg_lambda=5.0,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1,
        )

        candidates["lgbm_ordinal"] = {"preprocess_mode": "ordinal", "model": LGBMClassifier(**base_lgbm)}

        if HAS_CATEGORY_ENCODERS:
            candidates["lgbm_catboost_encoder"] = {"preprocess_mode": "ce_catboost", "model": LGBMClassifier(**base_lgbm)}
            candidates["lgbm_james_stein_encoder"] = {"preprocess_mode": "ce_james_stein", "model": LGBMClassifier(**base_lgbm)}
            candidates["lgbm_mestimate_encoder"] = {"preprocess_mode": "ce_m_estimate", "model": LGBMClassifier(**base_lgbm)}
            candidates["lgbm_target_encoder"] = {"preprocess_mode": "ce_target", "model": LGBMClassifier(**base_lgbm)}
            candidates["lgbm_count_encoder"] = {"preprocess_mode": "ce_count", "model": LGBMClassifier(**base_lgbm)}

    if HAS_XGBOOST:
        pos_rate = df_dev[TARGET_COL].mean()
        neg_rate = 1 - pos_rate
        scale_pos_weight = neg_rate / pos_rate if pos_rate > 0 else 1

        base_xgb = dict(
            objective="binary:logistic",
            eval_metric="aucpr",
            n_estimators=700,
            learning_rate=0.035,
            max_depth=5,
            min_child_weight=20,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.2,
            reg_lambda=5.0,
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            tree_method="hist",
        )

        candidates["xgb_ordinal"] = {"preprocess_mode": "ordinal", "model": XGBClassifier(**base_xgb)}

        if HAS_CATEGORY_ENCODERS:
            candidates["xgb_catboost_encoder"] = {"preprocess_mode": "ce_catboost", "model": XGBClassifier(**base_xgb)}
            candidates["xgb_james_stein_encoder"] = {"preprocess_mode": "ce_james_stein", "model": XGBClassifier(**base_xgb)}
            candidates["xgb_mestimate_encoder"] = {"preprocess_mode": "ce_m_estimate", "model": XGBClassifier(**base_xgb)}

    return candidates


RUN_MODE = "fast"  # "fast" ou "full"

all_candidates = make_model_candidates()

if RUN_MODE == "fast":
    priority = [
        "lgbm_catboost_encoder",
        "lgbm_ordinal",
        "xgb_catboost_encoder",
        "xgb_ordinal",
        "hgb_ordinal",
    ]
    model_candidates = {k: all_candidates[k] for k in priority if k in all_candidates}
else:
    model_candidates = all_candidates

print("Candidatos:")
for k, v in model_candidates.items():
    print(k, "|", v["preprocess_mode"])

if not model_candidates:
    raise RuntimeError("Nenhum candidato disponível.")

## 10. Walk-forward temporal

In [ ]:
def make_pipeline_for_candidate(candidate):
    preprocess = make_preprocessor(
        mode=candidate["preprocess_mode"],
        numeric_features=selected_numeric_features,
        categorical_features=selected_categorical_features,
        text_features=selected_text_features,
    )
    return Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", clone(candidate["model"])),
    ])


def evaluate_pipeline_on_fold(model_name, candidate, df, train_idx, val_idx):
    X_train = df.loc[train_idx, selected_features].copy()
    y_train = df.loc[train_idx, TARGET_COL].astype(int)
    X_val = df.loc[val_idx, selected_features].copy()
    y_val = df.loc[val_idx, TARGET_COL].astype(int)

    pipe = make_pipeline_for_candidate(candidate)
    pipe.fit(X_train, y_train)
    proba_perda_val = get_positive_class_proba(pipe, X_val)

    roc_auc_perda = roc_auc_score(y_val, proba_perda_val)
    pr_auc_perda = average_precision_score(y_val, proba_perda_val)

    best_thr, best_p, best_r, best_f1 = find_best_f1_threshold_perda(y_val, proba_perda_val)
    thr_05 = threshold_metrics_perda(y_val, proba_perda_val, threshold=0.5)

    result = {
        "model": model_name,
        "preprocess_mode": candidate["preprocess_mode"],
        "roc_auc_perda": roc_auc_perda,
        "pr_auc_perda": pr_auc_perda,
        "taxa_perda_val": y_val.mean(),
        "taxa_ganho_val": 1 - y_val.mean(),
        "best_f1_threshold_perda": best_thr,
        "best_f1_precision_perda": best_p,
        "best_f1_recall_perda": best_r,
        "best_f1_perda": best_f1,
        "precision_perda_t05": thr_05["precision_perda"],
        "recall_perda_t05": thr_05["recall_perda"],
        "f1_perda_t05": thr_05["f1_perda"],
        "pred_perda_rate_t05": thr_05["pred_perda_rate"],
    }

    topk_perda = topk_risco_perda_metrics(y_val.values, proba_perda_val)

    if VALUE_COL in df.columns:
        topk_financeiro = topk_prioridade_financeira_metrics(
            y_true=y_val.values,
            proba_perda=proba_perda_val,
            valor_ajuizado=df.loc[val_idx, VALUE_COL].values,
        )
    else:
        topk_financeiro = pd.DataFrame()

    return result, topk_perda, topk_financeiro


walk_forward_results = []
topk_perda_results = []
topk_financeiro_results = []

for model_name, candidate in model_candidates.items():
    print("=" * 100)
    print("Modelo:", model_name)
    for fold in folds:
        print("  Fold", fold["fold"])
        result, topk_perda, topk_financeiro = evaluate_pipeline_on_fold(
            model_name=model_name,
            candidate=candidate,
            df=df_dev,
            train_idx=fold["train_idx"],
            val_idx=fold["val_idx"],
        )

        result["fold"] = fold["fold"]
        result["qtd_train"] = fold["qtd_train"]
        result["qtd_val"] = fold["qtd_val"]
        result["train_end_date"] = fold["train_end_date"]
        result["val_start_date"] = fold["val_start_date"]
        result["val_end_date"] = fold["val_end_date"]
        walk_forward_results.append(result)

        topk_perda["model"] = model_name
        topk_perda["fold"] = fold["fold"]
        topk_perda_results.append(topk_perda)

        if not topk_financeiro.empty:
            topk_financeiro["model"] = model_name
            topk_financeiro["fold"] = fold["fold"]
            topk_financeiro_results.append(topk_financeiro)

walk_forward_results = pd.DataFrame(walk_forward_results)
topk_perda_results = pd.concat(topk_perda_results, ignore_index=True)
topk_financeiro_results = pd.concat(topk_financeiro_results, ignore_index=True) if topk_financeiro_results else pd.DataFrame()

save_report(walk_forward_results, "69_3c_lgbm_xgb_walk_forward_metrics.csv")
save_report(topk_perda_results, "70_3c_lgbm_xgb_walk_forward_topk_risco_perda.csv")

if not topk_financeiro_results.empty:
    save_report(topk_financeiro_results, "71_3c_lgbm_xgb_walk_forward_topk_prioridade_financeira.csv")

walk_forward_results.sort_values(["pr_auc_perda", "roc_auc_perda"], ascending=False)

## 11. Resumo por modelo

In [ ]:
model_summary = (
    walk_forward_results
    .groupby(["model", "preprocess_mode"])
    .agg(
        mean_roc_auc_perda=("roc_auc_perda", "mean"),
        std_roc_auc_perda=("roc_auc_perda", "std"),
        mean_pr_auc_perda=("pr_auc_perda", "mean"),
        std_pr_auc_perda=("pr_auc_perda", "std"),
        mean_taxa_perda_val=("taxa_perda_val", "mean"),
        mean_taxa_ganho_val=("taxa_ganho_val", "mean"),
        mean_best_f1_perda=("best_f1_perda", "mean"),
        mean_best_f1_threshold_perda=("best_f1_threshold_perda", "mean"),
        mean_precision_perda_t05=("precision_perda_t05", "mean"),
        mean_recall_perda_t05=("recall_perda_t05", "mean"),
        mean_f1_perda_t05=("f1_perda_t05", "mean"),
    )
    .reset_index()
    .sort_values("mean_pr_auc_perda", ascending=False)
)

save_report(model_summary, "72_3c_lgbm_xgb_model_summary.csv")
model_summary

## 12. Top-k risco de perda por modelo

In [ ]:
topk_perda_summary = (
    topk_perda_results
    .groupby(["model", "top_k"])
    .agg(
        mean_precision_perda_at_k=("precision_perda_at_k", "mean"),
        mean_recall_perda_at_k=("recall_perda_at_k", "mean"),
        mean_lift_perda_at_k=("lift_perda_at_k", "mean"),
        mean_taxa_perda_base=("taxa_perda_base", "mean"),
    )
    .reset_index()
    .sort_values(["top_k", "mean_precision_perda_at_k"], ascending=[True, False])
)

save_report(topk_perda_summary, "73_3c_lgbm_xgb_topk_risco_perda_summary.csv")
topk_perda_summary

## 13. Top-k prioridade financeira por modelo

In [ ]:
if not topk_financeiro_results.empty:
    topk_financeiro_summary = (
        topk_financeiro_results
        .groupby(["model", "top_k"])
        .agg(
            mean_precision_perda_at_k=("precision_perda_at_k", "mean"),
            mean_recall_perda_at_k=("recall_perda_at_k", "mean"),
            mean_lift_perda_at_k=("lift_perda_at_k", "mean"),
            mean_share_valor_ajuizado_total=("share_valor_ajuizado_total", "mean"),
            mean_share_valor_perdas_total=("share_valor_perdas_total", "mean"),
        )
        .reset_index()
        .sort_values(["top_k", "mean_share_valor_perdas_total"], ascending=[True, False])
    )
    save_report(topk_financeiro_summary, "74_3c_lgbm_xgb_topk_prioridade_financeira_summary.csv")
    display(topk_financeiro_summary)
else:
    print("Top-k financeiro não calculado.")

## 14. Treinar melhor modelo no Dev completo e avaliar Holdout

In [ ]:
best_model_name = model_summary.iloc[0]["model"]
best_candidate = model_candidates[best_model_name]

print("Melhor modelo por PR-AUC de perda:", best_model_name)
print("Preprocessamento:", best_candidate["preprocess_mode"])

X_dev = df_dev[selected_features].copy()
y_dev = df_dev[TARGET_COL].astype(int)
X_holdout = df_holdout[selected_features].copy()
y_holdout = df_holdout[TARGET_COL].astype(int)

best_pipeline = make_pipeline_for_candidate(best_candidate)
best_pipeline.fit(X_dev, y_dev)

proba_perda_holdout = get_positive_class_proba(best_pipeline, X_holdout)

holdout_roc_auc_perda = roc_auc_score(y_holdout, proba_perda_holdout)
holdout_pr_auc_perda = average_precision_score(y_holdout, proba_perda_holdout)

best_thr, best_p, best_r, best_f1 = find_best_f1_threshold_perda(y_holdout, proba_perda_holdout)
thr_05 = threshold_metrics_perda(y_holdout, proba_perda_holdout, threshold=0.5)

holdout_metrics = pd.DataFrame([{
    "model": best_model_name,
    "preprocess_mode": best_candidate["preprocess_mode"],
    "roc_auc_perda": holdout_roc_auc_perda,
    "pr_auc_perda": holdout_pr_auc_perda,
    "taxa_perda_holdout": y_holdout.mean(),
    "taxa_ganho_holdout": 1 - y_holdout.mean(),
    "best_f1_threshold_perda": best_thr,
    "best_f1_precision_perda": best_p,
    "best_f1_recall_perda": best_r,
    "best_f1_perda": best_f1,
    "precision_perda_t05": thr_05["precision_perda"],
    "recall_perda_t05": thr_05["recall_perda"],
    "f1_perda_t05": thr_05["f1_perda"],
    "pred_perda_rate_t05": thr_05["pred_perda_rate"],
    "qtd_dev": len(df_dev),
    "qtd_holdout": len(df_holdout),
    "qtd_features": len(selected_features),
    "use_text_features": USE_TEXT_FEATURES,
}])

save_report(holdout_metrics, "75_3c_lgbm_xgb_holdout_best_model_metrics.csv")
holdout_metrics

## 15. Holdout top-k risco de perda

In [ ]:
holdout_topk_perda = topk_risco_perda_metrics(
    y_true=y_holdout.values,
    proba_perda=proba_perda_holdout,
    ks=(0.01, 0.05, 0.10, 0.20),
)

holdout_topk_perda["model"] = best_model_name
save_report(holdout_topk_perda, "76_3c_lgbm_xgb_holdout_topk_risco_perda.csv")
holdout_topk_perda

## 16. Holdout top-k prioridade financeira

In [ ]:
if VALUE_COL in df_holdout.columns:
    holdout_topk_financeiro = topk_prioridade_financeira_metrics(
        y_true=y_holdout.values,
        proba_perda=proba_perda_holdout,
        valor_ajuizado=df_holdout[VALUE_COL],
        ks=(0.01, 0.05, 0.10, 0.20),
    )
    holdout_topk_financeiro["model"] = best_model_name
    save_report(holdout_topk_financeiro, "77_3c_lgbm_xgb_holdout_topk_prioridade_financeira.csv")
    display(holdout_topk_financeiro)
else:
    print(f"{VALUE_COL} não encontrado.")

## 17. Ranking do holdout

In [ ]:
ranking_cols = ["Numerodistribuicao", "Identificador", DATE_COL, TARGET_COL_LEGACY, TARGET_COL, VALUE_COL]
ranking_cols = [c for c in ranking_cols if c in df_holdout.columns]

ranking_holdout = df_holdout[ranking_cols].copy()
ranking_holdout["target_classe_real"] = ranking_holdout[TARGET_COL].map({0: "banco_ganhou", 1: "banco_perdeu"})
ranking_holdout["proba_banco_perdeu"] = proba_perda_holdout
ranking_holdout["score_risco_perda"] = proba_perda_holdout

if VALUE_COL in ranking_holdout.columns:
    ranking_holdout["score_prioridade_financeira"] = (
        ranking_holdout["score_risco_perda"] *
        pd.to_numeric(ranking_holdout[VALUE_COL], errors="coerce").fillna(0)
    )
else:
    ranking_holdout["score_prioridade_financeira"] = np.nan

ranking_holdout["rank_risco_perda"] = ranking_holdout["score_risco_perda"].rank(ascending=False, method="first").astype(int)
ranking_holdout["rank_prioridade_financeira"] = ranking_holdout["score_prioridade_financeira"].rank(ascending=False, method="first").astype(int)

ranking_risco = ranking_holdout.sort_values("score_risco_perda", ascending=False)
ranking_financeiro = ranking_holdout.sort_values("score_prioridade_financeira", ascending=False)

ranking_risco_path = PROCESSED_DIR / "ranking_holdout_risco_perda_3c_lgbm_xgb.parquet"
ranking_financeiro_path = PROCESSED_DIR / "ranking_holdout_prioridade_financeira_3c_lgbm_xgb.parquet"

ranking_risco.to_parquet(ranking_risco_path, index=False)
ranking_financeiro.to_parquet(ranking_financeiro_path, index=False)

save_report(ranking_risco.head(1000), "78_3c_lgbm_xgb_ranking_holdout_top1000_risco_perda.csv")
save_report(ranking_financeiro.head(1000), "79_3c_lgbm_xgb_ranking_holdout_top1000_prioridade_financeira.csv")

ranking_risco.head(20)

## 18. Classification report

In [ ]:
pred_perda_holdout_t05 = (proba_perda_holdout >= 0.5).astype(int)

print("Classification report — threshold 0.5")
print("Classe 0 = banco ganhou")
print("Classe 1 = banco perdeu")
print(classification_report(
    y_holdout,
    pred_perda_holdout_t05,
    target_names=["banco_ganhou", "banco_perdeu"]
))

print("Confusion matrix — threshold 0.5")
print("Linhas = real | Colunas = previsto")
print(confusion_matrix(y_holdout, pred_perda_holdout_t05))

## 19. Feature importance

In [ ]:
feature_importance = pd.DataFrame()

try:
    model_obj = best_pipeline.named_steps["model"]
    pre_obj = best_pipeline.named_steps["preprocess"]

    if hasattr(model_obj, "feature_importances_"):
        try:
            feature_names = pre_obj.get_feature_names_out()
        except Exception:
            feature_names = [f"feature_{i}" for i in range(len(model_obj.feature_importances_))]

        feature_importance = pd.DataFrame({
            "feature": feature_names,
            "importance": model_obj.feature_importances_,
        }).sort_values("importance", ascending=False)

        save_report(feature_importance, "80_3c_lgbm_xgb_feature_importance.csv")
        display(feature_importance.head(50))
    else:
        print("Modelo não possui feature_importances_.")
except Exception as e:
    print("Não foi possível extrair feature importance:", repr(e))

## 20. Comparação 3A vs 3B vs 3C

In [ ]:
baseline_3a = {
    "model": "3A_random_forest_onehot_tfidf",
    "holdout_pr_auc_perda": 0.466804,
    "holdout_roc_auc_perda": 0.77858,
    "top5_precision_perda": 0.603004,
    "top10_fin_share_valor_perdas": 0.502560,
}

baseline_3b = {
    "model": "3B_hgb_catboost_encoder",
    "holdout_pr_auc_perda": 0.434753,
    "holdout_roc_auc_perda": 0.769694,
    "top5_precision_perda": 0.525751,
    "top10_fin_share_valor_perdas": 0.509398,
}

current_top5 = holdout_topk_perda.loc[holdout_topk_perda["top_k"] == 0.05, "precision_perda_at_k"].iloc[0]

if VALUE_COL in df_holdout.columns:
    current_top10_fin = holdout_topk_financeiro.loc[
        holdout_topk_financeiro["top_k"] == 0.10,
        "share_valor_perdas_total"
    ].iloc[0]
else:
    current_top10_fin = np.nan

comparison_3a_3b_3c = pd.DataFrame([
    baseline_3a,
    baseline_3b,
    {
        "model": f"3C_{best_model_name}",
        "holdout_pr_auc_perda": holdout_pr_auc_perda,
        "holdout_roc_auc_perda": holdout_roc_auc_perda,
        "top5_precision_perda": current_top5,
        "top10_fin_share_valor_perdas": current_top10_fin,
    },
])

comparison_3a_3b_3c["delta_pr_auc_vs_3a"] = comparison_3a_3b_3c["holdout_pr_auc_perda"] - comparison_3a_3b_3c.loc[0, "holdout_pr_auc_perda"]
comparison_3a_3b_3c["delta_roc_auc_vs_3a"] = comparison_3a_3b_3c["holdout_roc_auc_perda"] - comparison_3a_3b_3c.loc[0, "holdout_roc_auc_perda"]
comparison_3a_3b_3c["delta_top5_precision_vs_3a"] = comparison_3a_3b_3c["top5_precision_perda"] - comparison_3a_3b_3c.loc[0, "top5_precision_perda"]
comparison_3a_3b_3c["delta_top10_fin_vs_3a"] = comparison_3a_3b_3c["top10_fin_share_valor_perdas"] - comparison_3a_3b_3c.loc[0, "top10_fin_share_valor_perdas"]

save_report(comparison_3a_3b_3c, "81_3c_lgbm_xgb_comparison_3a_3b_3c.csv")
comparison_3a_3b_3c

## 21. Summary final

In [ ]:
summary_step_3c = pd.DataFrame([{
    "target_semantica": "0=banco_ganhou | 1=banco_perdeu",
    "run_mode": RUN_MODE,
    "use_text_features": USE_TEXT_FEATURES,
    "has_lightgbm": HAS_LIGHTGBM,
    "has_xgboost": HAS_XGBOOST,
    "has_category_encoders": HAS_CATEGORY_ENCODERS,
    "best_model_3c_walk_forward": best_model_name,
    "best_model_3c_preprocess": best_candidate["preprocess_mode"],
    "best_model_3c_mean_pr_auc_perda_wf": model_summary.loc[model_summary["model"] == best_model_name, "mean_pr_auc_perda"].iloc[0],
    "best_model_3c_mean_roc_auc_perda_wf": model_summary.loc[model_summary["model"] == best_model_name, "mean_roc_auc_perda"].iloc[0],
    "holdout_pr_auc_perda": holdout_pr_auc_perda,
    "holdout_roc_auc_perda": holdout_roc_auc_perda,
    "taxa_perda_holdout": y_holdout.mean(),
    "taxa_ganho_holdout": 1 - y_holdout.mean(),
    "qtd_features": len(selected_features),
    "qtd_dev": len(df_dev),
    "qtd_holdout": len(df_holdout),
    "ranking_risco_holdout_path": str(ranking_risco_path),
    "ranking_financeiro_holdout_path": str(ranking_financeiro_path),
}])

save_report(summary_step_3c, "82_summary_step_3c_lgbm_xgb.csv")
summary_step_3c.T

# O que enviar na próxima iteração

Depois de rodar, envie:

1. `model_summary`
2. `holdout_metrics`
3. `holdout_topk_perda`
4. `holdout_topk_financeiro`
5. `comparison_3a_3b_3c`
6. `summary_step_3c.T`
7. Se existir, `feature_importance.head(30)`

## Leitura esperada

Se o melhor modelo for `lgbm_catboost_encoder` ou `xgb_catboost_encoder`, significa que o **encoder CatBoostEncoder** foi útil. Isso não significa que o CatBoost foi instalado.

O baseline 3A continua sendo o campeão até que o 3C supere:

```text
PR-AUC perda holdout
Top 5% risco perda
Top 10% financeiro
```